In [1]:
# ============================================================
# Konfiguration (zentral & leicht anpassbar)
# Alle Stellschrauben an einem Ort -> kein Suchen mehr quer durch den Code.
# ============================================================

# Reproduzierbarkeit
SEED = 42

# Pfade
DATA_PATH     = 'Datasets/mnist_train_800.csv'   # Kaggle "Digit Recognizer": Spalte 'label' + pixel0..pixel783
MODEL_PATH    = 'Modelle/digit_cnn_NextStep5_toDevice.pth'
METADATA_PATH = 'Modelle/metadata_digit_NextStep5_toDevice.json'

# Datensatz-Struktur
LABEL_COL  = 'label'   # Name der Zielspalte in der CSV
IMAGE_SIZE = 28        # Bilder sind 28x28
N_CHANNELS = 1         # Graustufen -> 1 Kanal

# Architektur (die eigentlichen Stellschrauben des CNN)
CONV1_FILTERS = 16     # Filter im 1. Conv-Block
CONV2_FILTERS = 32     # Filter im 2. Conv-Block
FC_HIDDEN     = 64     # Neuronen in der voll verbundenen Schicht

# Training
BATCH_SIZE    = 64
LEARNING_RATE = 0.001
MAX_EPOCHS    = 30
PATIENCE      = 5      # Epochen ohne Verbesserung bis zum Early-Stopping-Abbruch

# Normalisierung der Pixel: 'standard' | 'minmax'
#   'minmax'   -> nur /255 (Werte in [0, 1])
#   'standard' -> /255, danach (x - mean) / std mit Statistik AUS DEN TRAININGSDATEN
NORM_TYPE = 'standard'


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import joblib
import json
from torch.utils.data import TensorDataset, DataLoader
import copy
import random

# --- Reproduzierbarkeit: alle relevanten Zufallsquellen fixieren ---
# Ohne torch.manual_seed waeren Gewichts-Initialisierung und das Shuffeln im
# DataLoader bei jedem Lauf anders -> nicht reproduzierbare Ergebnisse.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Geraet auswaehlen: GPU (CUDA) > Apple-Silicon (MPS) > CPU ---
# Erst auf CPU laufen lassen, um zu spueren, wie zaeh ein CNN dort wird -
# danach zeigt dieser Block, wie man die vorhandene Hardware nutzt.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")  # Fuer MacBooks mit M1/M2/M3 Chips
else:
    device = torch.device("cpu")
print("Verwendetes Geraet:", device)

# 1. Laden des Datensatzes
df = pd.read_csv(DATA_PATH)

# 2. Features (X = 784 Pixel) und Target (y = Ziffer) trennen
X = df.drop(LABEL_COL, axis=1).values.astype('float32')
y = df[LABEL_COL].values

# Anders als bei Iris ist KEIN LabelEncoder noetig: die Labels sind bereits
# Ganzzahlen 0..9. Anzahl Klassen direkt aus den Daten ableiten.
num_classes = int(len(np.unique(y)))

# 3. Aufteilen des Datensatzes (Train / Val / Test) - beide Splits mit demselben SEED
#    stratify=y haelt die Ziffern-Verteilung in allen Teilmengen gleich.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train)

# --- Normalisierung der Pixelwerte ---
# Bei Bildern tritt die /255-Skalierung an die Stelle des sklearn-Scalers.
# Pixel liegen in [0, 255]; Skalierung ist essenziell fuer die Konvergenz.
PIXEL_MAX = 255.0
X_train = X_train / PIXEL_MAX
X_val   = X_val   / PIXEL_MAX
X_test  = X_test  / PIXEL_MAX

if NORM_TYPE == 'standard':
    # WICHTIG (Vermeidung von Data Leakage): mean/std NUR aus den Trainingsdaten!
    pixel_mean = float(X_train.mean())
    pixel_std  = float(X_train.std())
    X_train = (X_train - pixel_mean) / pixel_std
    X_val   = (X_val   - pixel_mean) / pixel_std
    X_test  = (X_test  - pixel_mean) / pixel_std
elif NORM_TYPE == 'minmax':
    # nur /255 -> Werte bereits in [0, 1]; mean/std als neutrale Platzhalter
    pixel_mean, pixel_std = 0.0, 1.0
else:
    raise ValueError(f"Unbekannter NORM_TYPE: {NORM_TYPE!r} (erlaubt: 'standard', 'minmax')")
# -----------------------------------------

# 4. Flachen 784er-Vektor in Bildform bringen: (N, Kanal, Hoehe, Breite)
#    Genau diese 4D-Form erwartet ein Conv2d-Layer.
X_train = X_train.reshape(-1, N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
X_val   = X_val.reshape(-1, N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
X_test  = X_test.reshape(-1, N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)

# Umwandeln in PyTorch-Tensoren
X_train = torch.from_numpy(X_train).float()
X_val   = torch.from_numpy(X_val).float()
X_test  = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_val   = torch.from_numpy(y_val).long()
y_test  = torch.from_numpy(y_test).long()

# Datasets + DataLoader. Anders als bei Iris werden hier AUCH Val und Test
# ueber Loader in Batches verarbeitet - sonst muessten tausende Bilder auf
# einmal durchs Netz (Speicher/Laufzeit).
train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)
test_dataset  = TensorDataset(X_test, y_test)

g = torch.Generator()           # Generator mit Seed -> reproduzierbares Shuffeln
g.manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=g)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# Groesse nach zweimaligem MaxPool2d(2): IMAGE_SIZE -> /2 -> /2
pooled     = IMAGE_SIZE // 4
flatten_dim = CONV2_FILTERS * pooled * pooled

# Definieren des 2D-CNN als Sequential-Modell (rein sequentiell, kein eigener Klassen-Code)
net = nn.Sequential(
    nn.Conv2d(N_CHANNELS, CONV1_FILTERS, kernel_size=3, padding=1),  # (1,28,28)  -> (16,28,28)
    nn.ReLU(),
    nn.MaxPool2d(2),                                                 # (16,28,28) -> (16,14,14)
    nn.Conv2d(CONV1_FILTERS, CONV2_FILTERS, kernel_size=3, padding=1),  # -> (32,14,14)
    nn.ReLU(),
    nn.MaxPool2d(2),                                                 # (32,14,14) -> (32,7,7)
    nn.Flatten(),                                                    # -> 32*7*7 = flatten_dim
    nn.Linear(flatten_dim, FC_HIDDEN),                               # voll verbundene Schicht
    nn.ReLU(),
    nn.Linear(FC_HIDDEN, num_classes)                               # Ausgabeschicht (num_classes Logits)
)

net = net.to(device)   # Modell aufs gewaehlte Geraet verschieben

# Verlustkriterium und Optimierer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=LEARNING_RATE)

# --- Setup fuer Early Stopping ---
patience_counter = 0          # Zaehler fuer Epochen ohne Verbesserung
best_val_loss = float('inf')  # bester bisher gesehener Validation Loss (als float)
best_model_weights = None     # hier speichern wir die besten Gewichte

# Historie (ideal fuer spaetere Plots, z.B. mit matplotlib)
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

# Trainieren des CNN
for epoch in range(MAX_EPOCHS):
    net.train()  # Trainingsmodus aktivieren
    kumulierter_train_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = net(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        kumulierter_train_loss += loss.item()

    durchschnittlicher_train_loss = kumulierter_train_loss / len(train_loader)
    history['train_loss'].append(durchschnittlicher_train_loss)

    # Validierung nach jeder Epoche (in Batches)
    net.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for val_X, val_y in val_loader:
            val_X, val_y = val_X.to(device), val_y.to(device)
            val_out = net(val_X)
            val_loss_sum += criterion(val_out, val_y).item()
            _, val_pred = torch.max(val_out, 1)
            val_correct += (val_pred == val_y).sum().item()
            val_total   += val_y.size(0)

    val_loss = val_loss_sum / len(val_loader)   # Durchschnitt ueber die Val-Batches (float)
    val_acc  = val_correct / val_total
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch:3d} | Loss: {durchschnittlicher_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

    # --- Early Stopping Logik ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_weights = copy.deepcopy(net.state_dict())
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"\n--- Early Stopping ausgeloest in Epoche {epoch+1} ---")
        print(f"Bester Validation Loss war: {best_val_loss:.4f}")
        break
    # ----------------------------------------------

# Beste Gewichte zurueckladen (sonst bliebe das Modell im Overfitting-Zustand!)
net.load_state_dict(best_model_weights)
print("\nBeste Modellgewichte wurden wiederhergestellt.")

# Auswertung auf den Testdaten (in Batches)
net.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for test_X, test_y in test_loader:
        test_X, test_y = test_X.to(device), test_y.to(device)
        test_out = net(test_X)
        _, test_pred = torch.max(test_out, 1)
        test_correct += (test_pred == test_y).sum().item()
        test_total   += test_y.size(0)
accuracy = test_correct / test_total
print('Testgenauigkeit: ', accuracy)

# --- Abspeichern: Modell + Metadaten ---
# Kein separater Encoder/Scaler noetig: Labels sind bereits numerisch und die
# Normalisierung besteht nur aus wenigen Zahlen, die wir in die Metadaten legen.
torch.save(net.state_dict(), MODEL_PATH)

metadata = {
    'label_col': LABEL_COL,
    'class_names': [str(c) for c in range(num_classes)],
    'architecture': {
        'image_size':    int(IMAGE_SIZE),
        'n_channels':    int(N_CHANNELS),
        'conv1_filters': int(CONV1_FILTERS),
        'conv2_filters': int(CONV2_FILTERS),
        'fc_hidden':     int(FC_HIDDEN),
        'num_classes':   int(num_classes),
        'flatten_dim':   int(flatten_dim),
    },
    'normalization': {
        'norm_type':  NORM_TYPE,
        'pixel_max':  float(PIXEL_MAX),
        'pixel_mean': float(pixel_mean),
        'pixel_std':  float(pixel_std),
    },
    'best_val_loss': float(best_val_loss),
    'test_accuracy': float(accuracy),
    'seed': SEED,
    'torch_version': torch.__version__,
}
with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print("Metadaten gespeichert:", METADATA_PATH)


Verwendetes Geraet: mps
Epoch   0 | Loss: 0.8475 | Val Loss: 0.3417 | Val Acc: 0.8977
Epoch   1 | Loss: 0.2684 | Val Loss: 0.2308 | Val Acc: 0.9320
Epoch   2 | Loss: 0.1752 | Val Loss: 0.1752 | Val Acc: 0.9500
Epoch   3 | Loss: 0.1241 | Val Loss: 0.1587 | Val Acc: 0.9508
Epoch   4 | Loss: 0.0942 | Val Loss: 0.1150 | Val Acc: 0.9656
Epoch   5 | Loss: 0.0790 | Val Loss: 0.1123 | Val Acc: 0.9703
Epoch   6 | Loss: 0.0587 | Val Loss: 0.1175 | Val Acc: 0.9688
Epoch   7 | Loss: 0.0428 | Val Loss: 0.1157 | Val Acc: 0.9664
Epoch   8 | Loss: 0.0355 | Val Loss: 0.0982 | Val Acc: 0.9750
Epoch   9 | Loss: 0.0333 | Val Loss: 0.1059 | Val Acc: 0.9680
Epoch  10 | Loss: 0.0289 | Val Loss: 0.0911 | Val Acc: 0.9734
Epoch  11 | Loss: 0.0163 | Val Loss: 0.1365 | Val Acc: 0.9648
Epoch  12 | Loss: 0.0111 | Val Loss: 0.1087 | Val Acc: 0.9742
Epoch  13 | Loss: 0.0086 | Val Loss: 0.1153 | Val Acc: 0.9727
Epoch  14 | Loss: 0.0062 | Val Loss: 0.1020 | Val Acc: 0.9750
Epoch  15 | Loss: 0.0065 | Val Loss: 0.1680 | 

In [3]:
# Wiederladen von Modell + Metadaten (Architektur aus den Metadaten rekonstruiert)
import torch
import torch.nn as nn
import json

# --- Geraet auswaehlen: GPU (CUDA) > Apple-Silicon (MPS) > CPU ---
# Erst auf CPU laufen lassen, um zu spueren, wie zaeh ein CNN dort wird -
# danach zeigt dieser Block, wie man die vorhandene Hardware nutzt.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")  # Fuer MacBooks mit M1/M2/M3 Chips
else:
    device = torch.device("cpu")
print("Verwendetes Geraet:", device)

# Metadaten ZUERST laden -> daraus Architektur und Normalisierung rekonstruieren
with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)
arch = metadata['architecture']

# WICHTIG: Erst die Architektur definieren, DANN die Gewichte laden.
pooled = arch['image_size'] // 4   # zwei MaxPool2d(2)
net = nn.Sequential(
    nn.Conv2d(arch['n_channels'], arch['conv1_filters'], kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(arch['conv1_filters'], arch['conv2_filters'], kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(arch['flatten_dim'], arch['fc_hidden']),
    nn.ReLU(),
    nn.Linear(arch['fc_hidden'], arch['num_classes'])
)
net.load_state_dict(torch.load(MODEL_PATH, map_location=device))
net = net.to(device)
net.eval()

print("Architektur:", arch)
print("Normalisierung:", metadata['normalization'])
print("Test-Genauigkeit beim Training:", metadata['test_accuracy'])


Verwendetes Geraet: mps
Architektur: {'image_size': 28, 'n_channels': 1, 'conv1_filters': 16, 'conv2_filters': 32, 'fc_hidden': 64, 'num_classes': 10, 'flatten_dim': 1568}
Normalisierung: {'norm_type': 'standard', 'pixel_max': 255.0, 'pixel_mean': 0.13198693096637726, 'pixel_std': 0.3093786835670471}
Test-Genauigkeit beim Training: 0.98125


In [4]:
# Vorhersage mit dem trainierten CNN
# Statt interaktivem input() ziehen wir ein paar Beispielzeilen aus der CSV.
# Reproduzierbar, skript-/batch-tauglich und gleich mit Soll-/Ist-Vergleich.
import numpy as np
import pandas as pd

arch = metadata['architecture']
norm = metadata['normalization']

# Beispiel-Datenpunkte ziehen (eine Zeile = ein 28x28-Bild als 784 Pixel)
df_demo = pd.read_csv(DATA_PATH).sample(5, random_state=SEED)
X_demo = df_demo.drop(metadata['label_col'], axis=1).values.astype('float32')
y_true = df_demo[metadata['label_col']].values

# EXAKT dieselbe Normalisierung wie im Training (Parameter aus den Metadaten)
X_demo = X_demo / norm['pixel_max']
X_demo = (X_demo - norm['pixel_mean']) / norm['pixel_std']
X_demo = X_demo.reshape(-1, arch['n_channels'], arch['image_size'], arch['image_size'])

inputs = torch.tensor(X_demo, dtype=torch.float32).to(device)

# Vorhersage treffen
with torch.no_grad():
    outputs = net(inputs)
    _, predicted = torch.max(outputs, 1)

for i, pred in enumerate(predicted):
    print(f"Bild {i}: Vorhergesagt = {pred.item()} | Tatsaechlich = {y_true[i]}")


Bild 0: Vorhergesagt = 2 | Tatsaechlich = 2
Bild 1: Vorhergesagt = 3 | Tatsaechlich = 3
Bild 2: Vorhergesagt = 2 | Tatsaechlich = 2
Bild 3: Vorhergesagt = 3 | Tatsaechlich = 3
Bild 4: Vorhergesagt = 5 | Tatsaechlich = 5


In [5]:
# Vorhersagen als Wahrscheinlichkeiten (nutzt 'inputs' aus der vorherigen Zelle)
import torch.nn.functional as F

with torch.no_grad():
    output = net(inputs)
    _, max_index = torch.max(output, 1)

# Klassenwahrscheinlichkeiten berechnen
probabilities = F.softmax(output, dim=1)

for i in range(len(inputs)):
    ziffer = max_index[i].item()
    konfidenz = float(probabilities[i, ziffer])
    print(f"Bild {i}: Ziffer = {ziffer} (Konfidenz {konfidenz:.4f})")
    print("   Verteilung:", [round(float(p), 3) for p in probabilities[i]])


Bild 0: Ziffer = 2 (Konfidenz 0.9687)
   Verteilung: [0.0, 0.0, 0.969, 0.0, 0.0, 0.0, 0.0, 0.0, 0.031, 0.0]
Bild 1: Ziffer = 3 (Konfidenz 0.9999)
   Verteilung: [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Bild 2: Ziffer = 2 (Konfidenz 0.8937)
   Verteilung: [0.0, 0.008, 0.894, 0.002, 0.0, 0.0, 0.0, 0.059, 0.037, 0.0]
Bild 3: Ziffer = 3 (Konfidenz 0.9999)
   Verteilung: [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Bild 4: Ziffer = 5 (Konfidenz 0.9999)
   Verteilung: [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]
